# 07 — Twitch Inference Benchmark

Benchmark the serving workload before we move from notebooks into production Python modules.

We compare:
- row-at-a-time inference
- batched inference
- logistic regression
- histogram gradient boosting
- PyTorch MLP
- TorchScript MLP
- end-to-end score + Top-K ranking latency

All benchmarks are CPU-first so they are reproducible without an NVIDIA GPU.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter
import joblib
import torch
from torch import nn

PROJECT_DIR = Path("Distributed ML Inference & Ranking")
DATA_DIR = PROJECT_DIR / "data/processed/twitch"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"

TEST_PATH = DATA_DIR / "twitch_test_ranking.csv.gz"

test = pd.read_csv(TEST_PATH)

ID_COLUMNS = ["user_id", "streamer"]
LABEL_COLUMN = "label"
FUTURE_ONLY_COLUMNS = ["future_interactions", "future_watch_minutes"]

FEATURE_COLUMNS = [
    c for c in test.columns
    if c not in ID_COLUMNS + [LABEL_COLUMN] + FUTURE_ONLY_COLUMNS
]

print("Test:", test.shape)
print("Features:", len(FEATURE_COLUMNS))

## 1. Load saved models

In [ ]:
logreg = joblib.load(
    ARTIFACT_DIR / "baseline_logistic_regression.joblib"
)

hgb_bundle = joblib.load(
    ARTIFACT_DIR / "baseline_hist_gradient_boosting.joblib"
)

neural_prep = joblib.load(
    ARTIFACT_DIR / "neural_ranker_preprocessing.joblib"
)

traced_model = torch.jit.load(
    str(ARTIFACT_DIR / "neural_ranker_traced.pt")
)

traced_model.eval()

print("Models loaded.")

## 2. Prepare benchmark arrays

In [ ]:
X_raw = test[FEATURE_COLUMNS]

X_hgb = hgb_bundle["imputer"].transform(X_raw)

X_neural = neural_prep["imputer"].transform(X_raw)
X_neural = neural_prep["scaler"].transform(X_neural)
X_neural = X_neural.astype(np.float32)

print(X_raw.shape)

## 3. Benchmark helper

In [ ]:
def latency_stats(samples_ms):
    values = np.asarray(samples_ms)

    return {
        "mean_ms": values.mean(),
        "p50_ms": np.percentile(values, 50),
        "p95_ms": np.percentile(values, 95),
        "p99_ms": np.percentile(values, 99),
    }


def benchmark_callable(
    name,
    fn,
    n_items,
    repeats=25,
    warmup=5,
):
    for _ in range(warmup):
        fn()

    samples = []

    for _ in range(repeats):
        start = perf_counter()
        fn()
        elapsed = perf_counter() - start
        samples.append(elapsed * 1000)

    stats = latency_stats(samples)
    stats["model"] = name
    stats["items"] = n_items

    mean_seconds = stats["mean_ms"] / 1000
    stats["throughput_items_per_sec"] = (
        n_items / mean_seconds
        if mean_seconds > 0
        else np.nan
    )

    return stats

## 4. Batched inference at different candidate counts

In [ ]:
CANDIDATE_COUNTS = [1, 10, 100, 500, 1000, 5000]
results = []

for n in CANDIDATE_COUNTS:
    n = min(n, len(test))

    X_raw_n = X_raw.iloc[:n]
    X_hgb_n = X_hgb[:n]
    X_neural_n = X_neural[:n]

    results.append(
        benchmark_callable(
            "logistic_regression",
            lambda X=X_raw_n: logreg.predict_proba(X)[:, 1],
            n,
        )
    )

    results.append(
        benchmark_callable(
            "hist_gradient_boosting",
            lambda X=X_hgb_n: hgb_bundle["model"].predict_proba(X)[:, 1],
            n,
        )
    )

    results.append(
        benchmark_callable(
            "torchscript_mlp",
            lambda X=X_neural_n: torch.sigmoid(
                traced_model(torch.from_numpy(X))
            ).detach().numpy(),
            n,
        )
    )

benchmark_df = pd.DataFrame(results)
benchmark_df

## 5. Row-at-a-time vs batch inference

In [ ]:
N_ROWS = min(1000, len(test))

X_loop_raw = X_raw.iloc[:N_ROWS]
X_loop_neural = X_neural[:N_ROWS]

def logreg_row_loop():
    scores = []

    for i in range(N_ROWS):
        score = logreg.predict_proba(
            X_loop_raw.iloc[[i]]
        )[0, 1]
        scores.append(score)

    return scores


def logreg_batch():
    return logreg.predict_proba(
        X_loop_raw
    )[:, 1]


def neural_row_loop():
    scores = []

    with torch.inference_mode():
        for i in range(N_ROWS):
            x = torch.from_numpy(
                X_loop_neural[i:i+1]
            )
            score = torch.sigmoid(
                traced_model(x)
            ).item()
            scores.append(score)

    return scores


def neural_batch():
    with torch.inference_mode():
        x = torch.from_numpy(
            X_loop_neural
        )
        return torch.sigmoid(
            traced_model(x)
        ).numpy()


loop_vs_batch = pd.DataFrame(
    [
        benchmark_callable(
            "logreg_row_loop",
            logreg_row_loop,
            N_ROWS,
            repeats=10,
            warmup=2,
        ),
        benchmark_callable(
            "logreg_batch",
            logreg_batch,
            N_ROWS,
            repeats=20,
            warmup=3,
        ),
        benchmark_callable(
            "neural_row_loop",
            neural_row_loop,
            N_ROWS,
            repeats=10,
            warmup=2,
        ),
        benchmark_callable(
            "neural_batch",
            neural_batch,
            N_ROWS,
            repeats=20,
            warmup=3,
        ),
    ]
)

loop_vs_batch

## 6. End-to-end rank request benchmark

In [ ]:
def rank_user_candidates(user_frame, top_k=10):
    X = user_frame[FEATURE_COLUMNS]

    Xn = neural_prep["imputer"].transform(X)
    Xn = neural_prep["scaler"].transform(Xn)
    Xn = Xn.astype(np.float32)

    with torch.inference_mode():
        scores = torch.sigmoid(
            traced_model(
                torch.from_numpy(Xn)
            )
        ).numpy()

    result = user_frame[
        ["streamer"]
    ].copy()

    result["score"] = scores

    return result.nlargest(
        min(top_k, len(result)),
        "score",
    )


candidate_sizes = (
    test.groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

benchmark_users = candidate_sizes.head(100).index

request_latencies = []

for user_id in benchmark_users:
    user_frame = test[
        test["user_id"] == user_id
    ]

    start = perf_counter()

    _ = rank_user_candidates(
        user_frame,
        top_k=10,
    )

    request_latencies.append(
        {
            "user_id": user_id,
            "candidate_count": len(user_frame),
            "latency_ms": (
                perf_counter() - start
            ) * 1000,
        }
    )

request_df = pd.DataFrame(
    request_latencies
)

request_df["latency_ms"].describe(
    percentiles=[0.50, 0.95, 0.99]
)

## 7. Save benchmark outputs

In [ ]:
BENCHMARK_DIR = PROJECT_DIR / "benchmarks"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

benchmark_df.to_csv(
    BENCHMARK_DIR / "batch_inference_benchmark.csv",
    index=False,
)

loop_vs_batch.to_csv(
    BENCHMARK_DIR / "row_vs_batch_benchmark.csv",
    index=False,
)

request_df.to_csv(
    BENCHMARK_DIR / "rank_request_latency.csv",
    index=False,
)

print("Saved benchmark results.")

## What comes after the notebooks

At this point the modeling/recommendation research phase is complete enough to move into production code.

The next stage should extract stable components into:

```text
src/
├── data/
├── features/
├── retrieval/
├── models/
├── inference/
├── ranking/
└── serving/
```

Then we build:
- model loading/versioning
- `/predict`
- `/rank`
- batch inference
- request queues
- Ray Serve
- load testing
- Redis
- Prometheus
- Docker/Kubernetes
- later Triton/GPU benchmarking